# Multimodal exercise: image + text -> dog coat category

**Goal.** Practice writing, from scratch, the three pieces of a training pipeline that takes **two input modalities** (an image and a short text caption) and predicts a single category:

1. a `Dataset` that returns `(image_tensor, text_tensor, label)`
2. a model with two small encoders (a tiny CNN for the image, an `EmbeddingBag` for the text) whose outputs are concatenated and fed to a classifier head
3. a training loop

**Data.** Only 10 images, each paired with a one-sentence caption. The label is the dog's coat color: `0 = light`, `1 = dark`. With 10 examples the point is *not* to build a model that generalizes -- it's to check that every piece of the pipeline (shapes, forward pass, loss, backward pass) is wired correctly. A correct pipeline should be able to drive the training loss on these 10 examples close to 0.

The image transforms are given to you (same ones used in the transfer-learning lab) -- you don't need to write those again.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

torch.manual_seed(0)

## 1. Data

Ten images live in `images/`; `captions.csv` has one row per image with columns `file, text, label` (`0 = light coat`, `1 = dark coat`).

In [ ]:
IMAGES_DIR = Path("images")

df = pd.read_csv("captions.csv")
df

## 2. Image transform (given)

Same normalization used in the transfer-learning lab. The images are already small (224x224), so we only resize defensively and skip augmentation -- with 10 images we care about correctness, not generalization.

In [ ]:
IMAGE_SIZE = 64  # small on purpose: this CNN is tiny and runs on CPU

image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

## 3. Text vocabulary and tokenizer (given)

A tiny whitespace tokenizer and a vocabulary built from the 10 captions. Captions are turned into a fixed-length sequence of token ids (padded with the `<pad>` id `0`) so that every caption tensor has the same shape and batching is a plain stack -- no custom `collate_fn` needed.

In [ ]:
MAX_LEN = 12

def simple_tokenize(text):
    return text.lower().replace(",", "").split()

vocab = {"<pad>": 0, "<unk>": 1}
for caption in df["text"]:
    for tok in simple_tokenize(caption):
        if tok not in vocab:
            vocab[tok] = len(vocab)

VOCAB_SIZE = len(vocab)
print(f"vocab size: {VOCAB_SIZE}")

def encode_caption(text, max_len=MAX_LEN):
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in simple_tokenize(text)]
    ids = ids[:max_len]
    ids = ids + [vocab["<pad>"]] * (max_len - len(ids))
    return torch.tensor(ids, dtype=torch.long)

## 4. Dataset

Write a `Dataset` that, for a given index, returns `(image_tensor, text_tensor, label)`:
- look up the row (`file`, `text`, `label`) for that index in `self.df`
- load the image with `PIL.Image.open(...).convert("RGB")` and apply `image_transform`
- encode the caption with `encode_caption(...)`
- return the label as a `torch.long` tensor (or plain `int`, the `DataLoader` will handle it)

In [ ]:
class DogCoatDataset(Dataset):
    def __init__(self, df, images_dir, transform):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        # TODO: return the number of examples
        raise NotImplementedError

    def __getitem__(self, idx):
        # TODO:
        #  1. look up the row (file, text, label) for this idx in self.df
        #     (self.df.iloc[idx])
        #  2. open the image, convert to RGB, apply self.transform
        #  3. encode the caption with encode_caption(...)
        #  4. return (image_tensor, text_tensor, label)
        raise NotImplementedError


dataset = DogCoatDataset(df, IMAGES_DIR, image_transform)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

images, texts, labels = next(iter(loader))
print(images.shape, texts.shape, labels.shape)
# expected: torch.Size([4, 3, 64, 64]) torch.Size([4, 12]) torch.Size([4])

## 5. Model

Three pieces:

- `SmallCNN`: a tiny CNN **from scratch** (no pretrained weights). Two `conv -> relu -> pool` blocks followed by a linear layer are enough; it should output a feature vector of size `image_embed_dim`.
- `TextEncoder`: a single `nn.EmbeddingBag(VOCAB_SIZE, text_embed_dim, mode="mean", padding_idx=0)` -- it takes the `(batch, MAX_LEN)` token tensor directly (no offsets needed in this 2D-input mode) and averages the embeddings of the tokens in each caption.
- `FusionClassifier`: concatenates the image and text feature vectors and applies one linear layer down to `num_classes` logits.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, image_embed_dim=32):
        super().__init__()
        # TODO: define 2 conv layers (e.g. 3->8 and 8->16 channels, kernel_size=3,
        # padding=1) and a pooling layer (e.g. nn.MaxPool2d(2)).
        # With IMAGE_SIZE=64 and two 2x2 pools, the spatial size becomes 16x16,
        # so the final linear layer's input size is 16 * 16 * 16 (channels * H * W).
        raise NotImplementedError

    def forward(self, x):
        # TODO: conv -> relu -> pool, twice, then flatten and apply the linear layer
        raise NotImplementedError


class TextEncoder(nn.Module):
    def __init__(self, vocab_size, text_embed_dim=16):
        super().__init__()
        # TODO: define a single nn.EmbeddingBag(vocab_size, text_embed_dim,
        # mode="mean", padding_idx=0)
        raise NotImplementedError

    def forward(self, text):
        # TODO: run the embedding bag on `text` (shape: batch x MAX_LEN) and return it
        raise NotImplementedError


class FusionClassifier(nn.Module):
    def __init__(self, image_embed_dim=32, text_embed_dim=16, num_classes=2):
        super().__init__()
        # TODO: create self.cnn (SmallCNN), self.text_encoder (TextEncoder),
        # and self.classifier: a nn.Linear from
        # (image_embed_dim + text_embed_dim) to num_classes
        raise NotImplementedError

    def forward(self, image, text):
        # TODO:
        #  1. get the image feature vector from self.cnn
        #  2. get the text feature vector from self.text_encoder
        #  3. concatenate them along dim=1
        #  4. pass the result through self.classifier and return the logits
        raise NotImplementedError


model = FusionClassifier()
logits = model(images, texts)
print(logits.shape)  # expected: torch.Size([4, 2])

## 6. Training loop

A standard loop: for each epoch, iterate over the `DataLoader`, compute the loss with `nn.CrossEntropyLoss`, backpropagate, and step the optimizer. Track and print the average loss and accuracy per epoch.

Since there are only 10 examples, training on all of them (no held-out set) and checking that the loss goes to (near) 0 and accuracy to 100% is the goal here -- it tells you the pipeline is wired correctly.

In [ ]:
model = FusionClassifier()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

num_epochs = 30
for epoch in range(num_epochs):
    total_loss = 0.0
    correct = 0
    total = 0
    for images_b, texts_b, labels_b in train_loader:
        # TODO:
        #  1. zero the optimizer's gradients
        #  2. forward pass: logits = model(images_b, texts_b)
        #  3. compute the loss with criterion
        #  4. backward pass + optimizer step
        raise NotImplementedError

        # once the above is implemented, this bookkeeping will work as-is:
        total_loss += loss.item() * labels_b.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels_b).sum().item()
        total += labels_b.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"epoch {epoch+1:3d}  loss={avg_loss:.4f}  accuracy={accuracy:.2f}")

## 7. Sanity check

If everything is wired correctly, the training accuracy should reach 100% (the model can memorize 10 examples). If it's stuck near 50%, check the shapes at each step first (`images.shape`, `texts.shape`, `logits.shape`) before suspecting the model itself.